In [ ]:
!pip install -q "timm>=1.0.9" grad-cam

## 1. Configuración

In [ ]:
import os, json, glob, zipfile, itertools
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy import stats
from sklearn.metrics import (classification_report, confusion_matrix, f1_score,
                             roc_auc_score, cohen_kappa_score, balanced_accuracy_score)
from sklearn.preprocessing import label_binarize


class CFG:
    INPUT_ROOT = "/kaggle/input"
    DATA_DIR   = "/kaggle/input/datasets/ibrahimagabardiop/niayes-crops-disease-v2"
    FROZEN_DIR = "/kaggle/input/datasets/emersonquispe/plantnova-particion-congelada"
    OUT_DIR    = "/kaggle/working"

    MODELO_PRINCIPAL = "convnext_small_cbam"
    MODELO_ABLACION  = "convnext_small_plain"

    N_BOOTSTRAP = 2000
    ALPHA       = 0.05        # intervalos al 95%
    N_GRADCAM   = 8
    SEED        = 42

FIG_DIR = os.path.join(CFG.OUT_DIR, "figuras"); os.makedirs(FIG_DIR, exist_ok=True)
TAB_DIR = os.path.join(CFG.OUT_DIR, "tablas");  os.makedirs(TAB_DIR, exist_ok=True)

rng = np.random.default_rng(CFG.SEED)
np.random.seed(CFG.SEED); torch.manual_seed(CFG.SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

mpl.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "savefig.bbox": "tight",
                     "font.size": 10, "axes.grid": True, "grid.alpha": 0.3})

def save_fig(fig, nombre):
    fig.savefig(os.path.join(FIG_DIR, nombre))
    print(f"  guardado: {nombre}")

ORDEN = ["convnext_small_cbam", "convnext_small_plain", "resnet50",
         "efficientnet_b3", "swin_tiny", "mobilenetv3_large"]
ETIQUETA = {"convnext_small_cbam": "ConvNeXt-S + CBAM", "convnext_small_plain": "ConvNeXt-S",
            "resnet50": "ResNet-50", "efficientnet_b3": "EfficientNet-B3",
            "swin_tiny": "Swin-T", "mobilenetv3_large": "MobileNetV3-L"}
print(f"Device: {device}")

## 2. Cargar los resultados de las seis cuentas

In [ ]:
def localizar(nombre_archivo):
    # Busca el archivo en cualquier dataset adjunto, sin depender del nombre exacto
    hits = glob.glob(os.path.join(CFG.INPUT_ROOT, "**", nombre_archivo), recursive=True)
    return hits

R = {}
for var in ORDEN:
    lt = [p for p in localizar("logits_test.npz") if var in p or var.replace("_", "-") in p]
    lv = [p for p in localizar("logits_val.npz")  if var in p or var.replace("_", "-") in p]
    cf = [p for p in localizar("config.json")     if var in p or var.replace("_", "-") in p]
    if not lt:
        print(f"  ausente: {var}")
        continue
    d_test = np.load(lt[0])
    entrada = {"logits_test": d_test["logits"].astype(np.float64),
               "y_test": d_test["labels"].astype(int)}
    if lv:
        d_val = np.load(lv[0])
        entrada["logits_val"] = d_val["logits"].astype(np.float64)
        entrada["y_val"] = d_val["labels"].astype(int)
    if cf:
        with open(cf[0]) as f:
            entrada["config"] = json.load(f)
    R[var] = entrada
    print(f"  cargado: {var:22s} test {entrada['logits_test'].shape}")

assert R, "No se encontró ningún resultado. Revisa que los datasets estén adjuntos."
VARIANTES = [v for v in ORDEN if v in R]

y_true = R[VARIANTES[0]]["y_test"]
for v in VARIANTES:
    assert np.array_equal(R[v]["y_test"], y_true), \
        f"{v} evaluó un orden de prueba distinto: las comparaciones pareadas no serían válidas"

cfg0 = R[VARIANTES[0]].get("config", {})
class_names = cfg0.get("datos", {}).get("class_names")
if class_names is None:
    from torchvision import datasets
    class_names = datasets.ImageFolder(os.path.join(CFG.DATA_DIR, "test")).classes
num_classes = len(class_names)

def short_name(raw):
    partes = raw.split("__")
    return partes[-1].replace("_", " ") if len(partes) > 1 else raw.replace("_", " ")

labels_cortos = [short_name(c) for c in class_names]
PRED = {v: R[v]["logits_test"].argmax(1) for v in VARIANTES}
print(f"\n{len(VARIANTES)} variantes | {len(y_true)} imágenes de prueba | {num_classes} clases")

## 3. Tabla comparativa de arquitecturas

In [ ]:
def softmax(z):
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

y_bin = label_binarize(y_true, classes=range(num_classes))

filas = []
for v in VARIANTES:
    p, c = PRED[v], R[v].get("config", {})
    rep = classification_report(y_true, p, output_dict=True, zero_division=0)
    pr = softmax(R[v]["logits_test"])
    res = c.get("resultado", {}) if c else {}
    lat = res.get("latencia", {})
    top3 = float(np.mean([t in np.argsort(-q)[:3] for q, t in zip(pr, y_true)]))
    try:
        auc = roc_auc_score(y_bin, pr, multi_class="ovr", average="macro")
    except Exception:
        auc = np.nan
    filas.append({
        "Arquitectura": ETIQUETA[v],
        "Accuracy": rep["accuracy"],
        "Macro-F1": rep["macro avg"]["f1-score"],
        "Weighted-F1": rep["weighted avg"]["f1-score"],
        "Macro-Precision": rep["macro avg"]["precision"],
        "Macro-Recall": rep["macro avg"]["recall"],
        "Balanced-Acc": balanced_accuracy_score(y_true, p),
        "Top-3 Acc": top3,
        "AUC macro (OvR)": auc,
        "Kappa": cohen_kappa_score(y_true, p),
        "Parámetros (M)": round(c.get("parametros", 0) / 1e6, 2) if c else np.nan,
        "GFLOPs": round(lat.get("gflops"), 2) if lat.get("gflops") else np.nan,
        "Latencia b=1 (ms)": round(lat.get("media_ms", np.nan), 1) if lat else np.nan,
        "Img/s (b=32)": round(lat.get("por_lote", {}).get("32", {})
                              .get("imagenes_por_s", np.nan), 0) if lat else np.nan,
        "Train limpio Acc": res.get("train_limpio_accuracy", np.nan),
        "Entrenamiento (min)": res.get("minutos_entrenamiento", np.nan),
        "Mejor época": res.get("mejor_epoca", np.nan),
    })

tabla_arq = pd.DataFrame(filas)
tabla_arq.to_csv(os.path.join(TAB_DIR, "tabla_comparativa_arquitecturas.csv"), index=False)
print(tabla_arq.round(4).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(tabla_arq)); w = 0.26
for i, (col, color) in enumerate([("Accuracy", "#2a6f97"), ("Macro-F1", "#e0a458"),
                                  ("Weighted-F1", "#8fbf9f")]):
    b = ax.bar(x + (i - 1) * w, tabla_arq[col], w, label=col, color=color)
    for r, val in zip(b, tabla_arq[col]):
        ax.text(r.get_x() + r.get_width() / 2, val + 0.002, f"{val:.3f}",
                ha="center", fontsize=6.5, rotation=90)
lo = min(0.90, tabla_arq[["Accuracy", "Macro-F1", "Weighted-F1"]].values.min() - 0.02)
ax.set_ylim(lo, 1.02); ax.set_xticks(x)
ax.set_xticklabels(tabla_arq["Arquitectura"], rotation=18, ha="right")
ax.set_ylabel("Valor"); ax.legend(fontsize=8); ax.grid(axis="x", visible=False)
ax.set_title("Comparación de arquitecturas sobre Niayes Crops Disease V2\n"
             "(misma partición, misma receta de entrenamiento)", fontweight="bold")
save_fig(fig, "fig_comparativa_arquitecturas.png"); plt.show()

## 4. Ablación de CBAM

In [ ]:
def mcnemar(pred_a, pred_b, y):
    ok_a, ok_b = pred_a == y, pred_b == y
    b = int(np.sum(ok_a & ~ok_b))
    c = int(np.sum(~ok_a & ok_b))
    n = len(y)
    if b + c == 0:
        return {"b": b, "c": c, "prueba": "sin discordantes", "estadistico": np.nan, "p": 1.0}
    if b + c < 25:
        p = float(stats.binomtest(b, b + c, 0.5).pvalue)
        return {"b": b, "c": c, "prueba": "binomial exacta", "estadistico": np.nan, "p": p}
    chi2 = (abs(b - c) - 1) ** 2 / (b + c)
    return {"b": b, "c": c, "prueba": "McNemar (χ², corregida)",
            "estadistico": float(chi2), "p": float(stats.chi2.sf(chi2, 1))}


def dif_accuracy_ic(pred_a, pred_b, y, alpha=CFG.ALPHA):
    # IC para la diferencia de proporciones pareadas
    ok_a, ok_b = pred_a == y, pred_b == y
    n = len(y)
    b = np.sum(ok_a & ~ok_b); c = np.sum(~ok_a & ok_b)
    d = (b - c) / n
    se = np.sqrt((b + c) - (b - c) ** 2 / n) / n
    z = stats.norm.ppf(1 - alpha / 2)
    return d, d - z * se, d + z * se


PRIN, ABL = CFG.MODELO_PRINCIPAL, CFG.MODELO_ABLACION
if PRIN in R and ABL in R:
    m = mcnemar(PRED[PRIN], PRED[ABL], y_true)
    d, lo_d, hi_d = dif_accuracy_ic(PRED[PRIN], PRED[ABL], y_true)
    acc_p = (PRED[PRIN] == y_true).mean(); acc_a = (PRED[ABL] == y_true).mean()
    f1_p = f1_score(y_true, PRED[PRIN], average="macro", zero_division=0)
    f1_a = f1_score(y_true, PRED[ABL],  average="macro", zero_division=0)

    print(f"ConvNeXt-S + CBAM : accuracy {acc_p:.4f} | macro-F1 {f1_p:.4f}")
    print(f"ConvNeXt-S        : accuracy {acc_a:.4f} | macro-F1 {f1_a:.4f}")
    print(f"\nDiscordantes: CBAM acierta y el otro falla = {m['b']}; al revés = {m['c']}")
    print(f"Prueba: {m['prueba']}")
    if np.isfinite(m["estadistico"]):
        print(f"Estadístico = {m['estadistico']:.4f}")
    print(f"p = {m['p']:.4g}")
    print(f"Diferencia de accuracy = {d:+.4f}  IC 95% [{lo_d:+.4f}, {hi_d:+.4f}]")

    if m["p"] < 0.05:
        signo = "a favor de CBAM" if d > 0 else "en contra de CBAM"
        print(f"\n→ Diferencia estadísticamente significativa (p < 0.05), {signo}.")
    else:
        print("\n→ La diferencia NO alcanza significancia estadística (p ≥ 0.05).")
        print("  Hay que reportarlo así: CBAM no muestra una mejora demostrable en")
        print("  este conjunto, y no se le pueden atribuir las mejoras del modelo.")
        print("  Un resultado nulo bien documentado es publicable; una atribución")
        print("  sin respaldo es lo que motivó la revisión mayor.")
    ablacion = {"accuracy_cbam": acc_p, "accuracy_sin_cbam": acc_a,
                "macro_f1_cbam": f1_p, "macro_f1_sin_cbam": f1_a,
                "diferencia_accuracy": d, "ic95": [lo_d, hi_d], **m}
else:
    ablacion = None
    print("Faltan las dos variantes ConvNeXt-Small: no se puede hacer la ablación")

### Significancia sobre macro-F1 y sobre el subconjunto sin fuga


In [ ]:
def bootstrap_pareado_macro_f1(pa, pb, y, n_boot=CFG.N_BOOTSTRAP, alpha=CFG.ALPHA):
    n, C = len(y), num_classes
    difs = np.empty(n_boot)
    r = np.random.default_rng(CFG.SEED)
    for b in range(n_boot):
        idx = r.integers(0, n, n)
        yt = y[idx]
        n_true = np.bincount(yt, minlength=C).astype(float)
        f1s = []
        for p in (pa[idx], pb[idx]):
            tp = np.bincount(yt[yt == p], minlength=C).astype(float)
            n_pred = np.bincount(p, minlength=C).astype(float)
            with np.errstate(divide="ignore", invalid="ignore"):
                pr_ = np.where(n_pred > 0, tp / n_pred, 0.0)
                rc_ = np.where(n_true > 0, tp / n_true, 0.0)
                f1_ = np.where(pr_ + rc_ > 0, 2 * pr_ * rc_ / (pr_ + rc_), 0.0)
            f1s.append(f1_[n_true > 0].mean())
        difs[b] = f1s[0] - f1s[1]
    lo = np.percentile(difs, 100 * alpha / 2)
    hi = np.percentile(difs, 100 * (1 - alpha / 2))
    p_val = 2 * min((difs <= 0).mean(), (difs >= 0).mean())
    return float(difs.mean()), float(lo), float(hi), float(min(1.0, p_val))


if PRIN in R and ABL in R:
    d_f1, lo_f1, hi_f1, p_f1 = bootstrap_pareado_macro_f1(PRED[PRIN], PRED[ABL], y_true)
    print(f"Diferencia de macro-F1 (CBAM − sin CBAM) = {d_f1:+.4f}")
    print(f"  IC 95% bootstrap [{lo_f1:+.4f}, {hi_f1:+.4f}] | p = {p_f1:.4g}")
    if lo_f1 > 0:
        print("  → El intervalo no cruza cero: la mejora en macro-F1 es consistente.")
    else:
        print("  → El intervalo CRUZA CERO: no se puede afirmar mejora en macro-F1.")
    if ablacion is not None:
        ablacion.update({"macro_f1_dif": d_f1, "macro_f1_ic95": [lo_f1, hi_f1],
                         "macro_f1_p": p_f1})

In [ ]:
_mask = os.path.join(CFG.FROZEN_DIR, "clean_test_mask.npy")
if PRIN in R and ABL in R and os.path.exists(_mask):
    cl = np.load(_mask)
    m_cl = mcnemar(PRED[PRIN][cl], PRED[ABL][cl], y_true[cl])
    d_cl, lo_cl, hi_cl = dif_accuracy_ic(PRED[PRIN][cl], PRED[ABL][cl], y_true[cl])
    print(f"Ablación sobre el subconjunto SIN FUGA ({int(cl.sum())} imágenes)")
    print(f"  discordantes b={m_cl['b']} c={m_cl['c']} | {m_cl['prueba']} | p = {m_cl['p']:.4g}")
    print(f"  Δ accuracy = {d_cl:+.4f}  IC 95% [{lo_cl:+.4f}, {hi_cl:+.4f}]")
    if ablacion is not None:
        ablacion["sin_fuga"] = {"n": int(cl.sum()), "p": m_cl["p"],
                                "dif_accuracy": d_cl, "ic95": [lo_cl, hi_cl]}
    if m_cl["p"] >= 0.05 and ablacion and ablacion.get("p", 1) < 0.05:
        print("\n  ATENCIÓN: la ventaja de CBAM es significativa en la prueba completa")
        print("  pero NO en el subconjunto sin fuga. Hay que reportar ambas cifras y")
        print("  apoyar la conclusión en la segunda, que es la menos contaminada.")

In [ ]:
# Cada variante contra el modelo principal
if PRIN in R:
    filas = []
    for v in VARIANTES:
        if v == PRIN:
            continue
        m = mcnemar(PRED[PRIN], PRED[v], y_true)
        d, lo_d, hi_d = dif_accuracy_ic(PRED[PRIN], PRED[v], y_true)
        filas.append({
            "Comparación": f"{ETIQUETA[PRIN]} vs {ETIQUETA[v]}",
            "Δ Accuracy": d, "IC 95% inf": lo_d, "IC 95% sup": hi_d,
            "b": m["b"], "c": m["c"], "Prueba": m["prueba"],
            "Estadístico": m["estadistico"], "p": m["p"],
            "Significativo (α=0.05)": "Sí" if m["p"] < 0.05 else "No",
        })
    tabla_mcnemar = pd.DataFrame(filas)
    tabla_mcnemar.to_csv(os.path.join(TAB_DIR, "tabla_mcnemar.csv"), index=False)
    print(tabla_mcnemar.round(5).to_string(index=False))

## 5. Evaluación sin fuga de datos

In [ ]:
mask_path = os.path.join(CFG.FROZEN_DIR, "clean_test_mask.npy")
if os.path.exists(mask_path):
    clean = np.load(mask_path)
    assert len(clean) == len(y_true), "La máscara no coincide con el conjunto de prueba"
    filas = []
    for v in VARIANTES:
        p = PRED[v]
        filas.append({
            "Arquitectura": ETIQUETA[v],
            "Acc (prueba completa)": (p == y_true).mean(),
            "Acc (sin fuga)": (p[clean] == y_true[clean]).mean(),
            "Macro-F1 (completa)": f1_score(y_true, p, average="macro", zero_division=0),
            "Macro-F1 (sin fuga)": f1_score(y_true[clean], p[clean],
                                            average="macro", zero_division=0),
        })
    tabla_fuga = pd.DataFrame(filas)
    tabla_fuga["Δ Accuracy"] = (tabla_fuga["Acc (sin fuga)"]
                                - tabla_fuga["Acc (prueba completa)"])
    tabla_fuga.to_csv(os.path.join(TAB_DIR, "tabla_sin_fuga.csv"), index=False)
    print(f"Prueba completa: {len(y_true)} | sin fuga: {int(clean.sum())} "
          f"({clean.mean():.1%})\n")
    print(tabla_fuga.round(4).to_string(index=False))
    print("\nLa columna Δ es la caída atribuible a la fuga. En el artículo debe decir")
    print("«subconjunto con duplicados detectables retirados», no «estimación no sesgada»:")
    print("ningún método garantiza encontrar todas las variantes de una fotografía.")
else:
    clean = None
    print(f"No se encontró {mask_path}: se omite la evaluación sin fuga")

In [ ]:
if clean is not None:
    p_ = PRED[PRIN] if PRIN in R else PRED[VARIANTES[0]]
    f1_full  = f1_score(y_true, p_, average=None,
                        labels=range(num_classes), zero_division=0)
    f1_clean = f1_score(y_true[clean], p_[clean], average=None,
                        labels=range(num_classes), zero_division=0)
    n_full  = np.bincount(y_true, minlength=num_classes)
    n_clean = np.bincount(y_true[clean], minlength=num_classes)

    cambio = pd.DataFrame({
        "Clase": labels_cortos,
        "n completa": n_full, "n sin fuga": n_clean,
        "% retirado": 1 - n_clean / np.maximum(1, n_full),
        "F1 completa": f1_full, "F1 sin fuga": f1_clean,
        "Δ F1": f1_clean - f1_full,
    }).sort_values("Δ F1")
    cambio.to_csv(os.path.join(TAB_DIR, "tabla_cambio_por_clase_sin_fuga.csv"), index=False)

    print("Clases que MÁS pierden al retirar duplicados:")
    print(cambio.head(8).round(4).to_string(index=False))
    print("\nClases que más ganan (suele ser ruido de muestreo en clases pequeñas):")
    print(cambio.tail(4).round(4).to_string(index=False))

    muy_contaminadas = cambio[cambio["% retirado"] > 0.5]
    if len(muy_contaminadas):
        print(f"\n{len(muy_contaminadas)} clases pierden más de la mitad de su conjunto")
        print("de prueba al retirar duplicados. Sus métricas sin fuga descansan en muy")
        print("pocas imágenes y hay que decirlo al reportarlas.")

## 6. Incertidumbre por clase

In [ ]:
def wilson(k, n, alpha=CFG.ALPHA):
    if n == 0:
        return (np.nan, np.nan)
    z = stats.norm.ppf(1 - alpha / 2)
    p = k / n
    den = 1 + z**2 / n
    centro = (p + z**2 / (2*n)) / den
    medio = z * np.sqrt(p*(1-p)/n + z**2/(4*n**2)) / den
    return (max(0.0, centro - medio), min(1.0, centro + medio))


def bootstrap_f1(y, p, n_boot=CFG.N_BOOTSTRAP, alpha=CFG.ALPHA):
    n, C = len(y), num_classes
    f1s = np.empty((n_boot, C)); macros = np.empty(n_boot)
    for b in range(n_boot):
        idx = rng.integers(0, n, n)
        yt, yp = y[idx], p[idx]
        tp = np.bincount(yt[yt == yp], minlength=C).astype(float)
        n_pred = np.bincount(yp, minlength=C).astype(float)
        n_true = np.bincount(yt, minlength=C).astype(float)
        with np.errstate(divide="ignore", invalid="ignore"):
            prec = np.where(n_pred > 0, tp / n_pred, 0.0)
            rec  = np.where(n_true > 0, tp / n_true, 0.0)
            f1   = np.where(prec + rec > 0, 2 * prec * rec / (prec + rec), 0.0)
        f1s[b] = f1
        macros[b] = f1[n_true > 0].mean()
    lo = np.percentile(f1s, 100 * alpha / 2, axis=0)
    hi = np.percentile(f1s, 100 * (1 - alpha / 2), axis=0)
    return lo, hi, (np.percentile(macros, 100 * alpha / 2),
                    np.percentile(macros, 100 * (1 - alpha / 2)))


p_prin = PRED[PRIN] if PRIN in R else PRED[VARIANTES[0]]
nombre_prin = ETIQUETA[PRIN] if PRIN in R else ETIQUETA[VARIANTES[0]]

cm = confusion_matrix(y_true, p_prin, labels=range(num_classes))
tp = np.diag(cm).astype(float)
n_true = cm.sum(axis=1).astype(float)
n_pred = cm.sum(axis=0).astype(float)

f1_lo, f1_hi, macro_ic = bootstrap_f1(y_true, p_prin)

filas = []
for c in range(num_classes):
    rec_lo, rec_hi = wilson(tp[c], n_true[c])
    pre_lo, pre_hi = wilson(tp[c], n_pred[c])
    prec = tp[c] / n_pred[c] if n_pred[c] else 0.0
    rec  = tp[c] / n_true[c] if n_true[c] else 0.0
    f1c  = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
    filas.append({
        "Clase": labels_cortos[c], "n prueba": int(n_true[c]), "Aciertos": int(tp[c]),
        "Precision": prec, "Precision IC95": f"[{pre_lo:.3f}, {pre_hi:.3f}]",
        "Recall": rec,     "Recall IC95":    f"[{rec_lo:.3f}, {rec_hi:.3f}]",
        "F1": f1c,         "F1 IC95":        f"[{f1_lo[c]:.3f}, {f1_hi[c]:.3f}]",
        "Amplitud IC F1": f1_hi[c] - f1_lo[c],
    })
tabla_clase = pd.DataFrame(filas).sort_values("Amplitud IC F1", ascending=False)
tabla_clase.to_csv(os.path.join(TAB_DIR, "tabla_incertidumbre_por_clase.csv"), index=False)

macro = f1_score(y_true, p_prin, average="macro", zero_division=0)
print(f"{nombre_prin} — macro-F1 {macro:.4f}  IC 95% [{macro_ic[0]:.4f}, {macro_ic[1]:.4f}]\n")
print("Clases con mayor incertidumbre:")
print(tabla_clase.head(8)[["Clase", "n prueba", "Aciertos", "Recall",
                           "Recall IC95", "F1", "F1 IC95"]].to_string(index=False))
print("\nEn el artículo: no concluir «validación robusta en las 36 clases» a partir de")
print("métricas puntuales. Las clases con pocas muestras tienen intervalos muy anchos.")

In [ ]:
d = tabla_clase.sort_values("F1")
fig, ax = plt.subplots(figsize=(9.5, 10))
y = np.arange(len(d))
lo = [float(s.strip("[]").split(",")[0]) for s in d["F1 IC95"]]
hi = [float(s.strip("[]").split(",")[1]) for s in d["F1 IC95"]]
err = np.vstack([d["F1"].values - np.array(lo), np.array(hi) - d["F1"].values])
colores = ["#c75146" if n < 50 else "#e0a458" if n < 200 else "#2a6f97"
           for n in d["n prueba"]]
ax.errorbar(d["F1"], y, xerr=err, fmt="none", ecolor="#999999", elinewidth=1, capsize=2)
ax.scatter(d["F1"], y, c=colores, s=26, zorder=3)
ax.set_yticks(y); ax.set_yticklabels(d["Clase"], fontsize=7)
ax.set_xlabel("F1-score (punto) con intervalo de confianza bootstrap al 95%")
ax.set_xlim(max(0, min(lo) - 0.03), 1.01)
ax.set_title(f"F1 por clase con incertidumbre — {nombre_prin}\n"
             "rojo: menos de 50 imágenes de prueba · naranja: menos de 200",
             fontweight="bold")
ax.grid(axis="y", visible=False)
for i, (f, n) in enumerate(zip(d["F1"], d["n prueba"])):
    ax.text(1.012, i, f"n={n}", va="center", fontsize=6)
save_fig(fig, "fig_f1_con_intervalos.png"); plt.show()

In [ ]:
for umbral in (0.70, 0.90):
    bajo_punto = sum(1 for _, r in tabla_clase.iterrows() if r["F1"] < umbral)
    bajo_ic = sum(1 for s_ in tabla_clase["F1 IC95"]
                  if float(s_.strip("[]").split(",")[0]) < umbral)
    print(f"Umbral {umbral:.2f} — clases con F1 puntual por debajo: {bajo_punto}; "
          f"con límite inferior del IC por debajo: {bajo_ic}")

print("\nLa segunda cifra es la defendible. Decir «las 36 clases superan 0.70»")
print("usando solo el valor puntual ignora la incertidumbre de las clases pequeñas,")
print("que es justo lo que objetan los revisores 3 y 5.")
print("\nAdemás hay que declarar si el 0.70 se fijó ANTES de evaluar o después de")
print("ver los resultados: el Revisor 3 lo pregunta de forma directa.")

In [ ]:
if PRIN in R and ABL in R:
    f1_c = f1_score(y_true, PRED[PRIN], average=None,
                    labels=range(num_classes), zero_division=0)
    f1_s = f1_score(y_true, PRED[ABL],  average=None,
                    labels=range(num_classes), zero_division=0)
    n_c = np.bincount(y_true, minlength=num_classes)
    comp = pd.DataFrame({"Clase": labels_cortos, "n": n_c,
                         "F1 con CBAM": f1_c, "F1 sin CBAM": f1_s,
                         "Δ F1": f1_c - f1_s}).sort_values("Δ F1", ascending=False)
    comp.to_csv(os.path.join(TAB_DIR, "tabla_ablacion_por_clase.csv"), index=False)

    mejora = int((comp["Δ F1"] > 1e-6).sum())
    empeora = int((comp["Δ F1"] < -1e-6).sum())
    igual = num_classes - mejora - empeora
    print(f"CBAM mejora {mejora} clases, empeora {empeora} y deja igual {igual}.")
    print("\nMayores mejoras:"); print(comp.head(5).round(4).to_string(index=False))
    print("\nMayores retrocesos:"); print(comp.tail(5).round(4).to_string(index=False))

    fig, ax = plt.subplots(figsize=(9.5, 9))
    d_ = comp.sort_values("Δ F1")
    colores = ["#c75146" if v < 0 else "#2a6f97" for v in d_["Δ F1"]]
    ax.barh(np.arange(len(d_)), d_["Δ F1"], color=colores)
    ax.axvline(0, c="#333333", lw=1)
    ax.set_yticks(np.arange(len(d_))); ax.set_yticklabels(d_["Clase"], fontsize=7)
    ax.set_xlabel("Δ F1 (con CBAM − sin CBAM)")
    ax.set_title("Efecto de CBAM clase por clase", fontweight="bold")
    ax.grid(axis="y", visible=False)
    save_fig(fig, "fig_ablacion_por_clase.png"); plt.show()

## 7. Matriz de confusión y errores principales

In [ ]:
cm_norm = confusion_matrix(y_true, p_prin, normalize="true")
pd.DataFrame(cm, index=class_names, columns=class_names).to_csv(
    os.path.join(TAB_DIR, "matriz_confusion.csv"))

fig, ax = plt.subplots(figsize=(13.5, 11.5))
sns.heatmap(cm_norm, cmap="Blues", vmin=0, vmax=1, square=True,
            linewidths=0.3, linecolor="#eeeeee",
            xticklabels=labels_cortos, yticklabels=labels_cortos, ax=ax,
            cbar_kws={"label": "Proporción de la clase real", "shrink": 0.7})
ax.set_xlabel("Clase predicha"); ax.set_ylabel("Clase real")
ax.set_title(f"Matriz de confusión normalizada — {nombre_prin}", fontweight="bold")
plt.xticks(rotation=90, fontsize=6.5); plt.yticks(rotation=0, fontsize=6.5)
ax.grid(False)
save_fig(fig, "fig_matriz_confusion.png"); plt.show()

In [ ]:
err = cm.copy(); np.fill_diagonal(err, 0)
pares = [(class_names[i], class_names[j], int(err[i, j]), err[i, j] / cm[i].sum())
         for i, j in zip(*np.nonzero(err))]
conf_df = (pd.DataFrame(pares, columns=["real", "predicha", "casos", "tasa"])
           .sort_values("casos", ascending=False).head(15).reset_index(drop=True))
conf_df.to_csv(os.path.join(TAB_DIR, "principales_confusiones.csv"), index=False)

if len(conf_df):
    etiquetas = [f"{short_name(r)} → {short_name(p)}"
                 for r, p in zip(conf_df["real"], conf_df["predicha"])]
    fig, ax = plt.subplots(figsize=(10, 0.45 * len(conf_df) + 1.8))
    barras = ax.barh(range(len(conf_df))[::-1], conf_df["casos"], color="#c75146")
    ax.set_yticks(range(len(conf_df))[::-1]); ax.set_yticklabels(etiquetas, fontsize=8)
    ax.set_xlabel("Imágenes mal clasificadas")
    ax.set_title("Principales confusiones (clase real → clase predicha)", fontweight="bold")
    for b, (c_, t_) in zip(barras, zip(conf_df["casos"], conf_df["tasa"])):
        ax.text(b.get_width() + max(conf_df["casos"]) * 0.01,
                b.get_y() + b.get_height() / 2, f"{c_} ({t_:.1%})",
                va="center", fontsize=7.5)
    ax.grid(axis="y", visible=False)
    save_fig(fig, "fig_principales_confusiones.png"); plt.show()
    print(conf_df.to_string(index=False))

## 8. Calibración y umbrales

In [ ]:
def ece(probs, y, n_bins=15):
    conf, pred = probs.max(1), probs.argmax(1)
    acc = (pred == y).astype(float)
    bordes = np.linspace(0, 1, n_bins + 1)
    e = 0.0
    for lo_, hi_ in zip(bordes[:-1], bordes[1:]):
        m = (conf > lo_) & (conf <= hi_)
        if m.any():
            e += m.mean() * abs(acc[m].mean() - conf[m].mean())
    return e


def ajustar_temperatura(logits, y):
    lg = torch.tensor(logits, dtype=torch.float32)
    yt = torch.tensor(y, dtype=torch.long)
    log_t = torch.zeros(1, requires_grad=True)
    opt = torch.optim.LBFGS([log_t], lr=0.1, max_iter=100)
    def paso():
        opt.zero_grad()
        p = F.cross_entropy(lg / log_t.exp(), yt)
        p.backward(); return p
    opt.step(paso)
    return float(log_t.exp().item())


TEMPERATURA = 1.0
if "logits_val" in R.get(PRIN, {}):
    TEMPERATURA = ajustar_temperatura(R[PRIN]["logits_val"], R[PRIN]["y_val"])
    pr_sin = softmax(R[PRIN]["logits_test"])
    pr_con = softmax(R[PRIN]["logits_test"] / TEMPERATURA)
    e_sin, e_con = ece(pr_sin, y_true), ece(pr_con, y_true)
    print(f"Temperatura ajustada en validación: {TEMPERATURA:.3f}")
    print(f"ECE sin calibrar : {e_sin:.4f}")
    print(f"ECE calibrado    : {e_con:.4f}")

    if e_con > e_sin:
        print("\nLa calibración EMPEORA el ECE en prueba. Pasa cuando validación y")
        print("prueba no se comportan igual. Reportarlo y dejar T = 1: aplicar una")
        print("temperatura que empeora la calibración sería un error metodológico.")
        TEMPERATURA = 1.0
        pr_con, e_con = pr_sin, e_sin
    elif e_sin > 0.02:
        print("\nEl ECE sin calibrar supera 0.02: la confianza del softmax NO era una")
        print("probabilidad fiable, así que el umbral de 0.60 del manuscrito carecía")
        print("de justificación. Con la temperatura ajustada ya puede defenderse.")
    else:
        print("\nEl modelo ya estaba razonablemente calibrado (ECE < 0.02), lo que")
        print("respalda presentar la confianza como probabilidad y justificar el umbral.")
else:
    pr_con = softmax(R[VARIANTES[0]]["logits_test"]); e_sin = e_con = np.nan
    print("Sin logits de validación: no se puede calibrar")

In [ ]:
if np.isfinite(e_sin):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for ax, pr, titulo, val in [(axes[0], pr_sin, "Sin calibrar", e_sin),
                                (axes[1], pr_con, f"Calibrado (T={TEMPERATURA:.2f})", e_con)]:
        conf, pred = pr.max(1), pr.argmax(1)
        acc = (pred == y_true).astype(float)
        bordes = np.linspace(0, 1, 16)
        xs, ys, ws = [], [], []
        for lo_, hi_ in zip(bordes[:-1], bordes[1:]):
            m = (conf > lo_) & (conf <= hi_)
            if m.any():
                xs.append(conf[m].mean()); ys.append(acc[m].mean()); ws.append(m.mean())
        ax.plot([0, 1], [0, 1], "--", c="gray", lw=1, label="Calibración perfecta")
        ax.scatter(xs, ys, s=np.array(ws) * 800 + 10, color="#2a6f97", alpha=0.75,
                   label="Observado (tamaño = proporción)")
        ax.set_xlabel("Confianza media"); ax.set_ylabel("Exactitud observada")
        ax.set_title(f"{titulo}\nECE = {val:.4f}", fontweight="bold")
        ax.set_xlim(0, 1.02); ax.set_ylim(0, 1.02); ax.legend(fontsize=8, loc="upper left")
    fig.suptitle("Diagrama de fiabilidad — conjunto de prueba", fontweight="bold")
    save_fig(fig, "fig_calibracion.png"); plt.show()

In [ ]:
# Curva cobertura-precision: justifica los umbrales en vez de fijarlos a mano
conf = pr_con.max(1); pred = pr_con.argmax(1); ok = pred == y_true
umbrales = np.round(np.arange(0.0, 1.0, 0.05), 2)
curva = pd.DataFrame({
    "Umbral": umbrales,
    "Cobertura": [(conf >= t).mean() for t in umbrales],
    "Precisión sobre aceptadas": [ok[conf >= t].mean() if (conf >= t).any() else np.nan
                                  for t in umbrales],
    "Rechazadas": [int((conf < t).sum()) for t in umbrales],
})
curva.to_csv(os.path.join(TAB_DIR, "curva_umbral.csv"), index=False)
print(curva.round(4).to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(curva["Cobertura"], curva["Precisión sobre aceptadas"], "o-", color="#2a6f97")
for _, r in curva.iloc[::2].iterrows():
    if np.isfinite(r["Precisión sobre aceptadas"]):
        ax.annotate(f"{r['Umbral']:.2f}", (r["Cobertura"], r["Precisión sobre aceptadas"]),
                    textcoords="offset points", xytext=(5, -10), fontsize=7, color="gray")
for t, color, etq in [(0.60, "#c75146", "0.60 — umbral del manuscrito"),
                      (0.70, "#e0a458", "0.70 — mínimo por clase")]:
    f = curva[curva["Umbral"] == t]
    if len(f):
        ax.scatter(f["Cobertura"], f["Precisión sobre aceptadas"], s=140,
                   facecolors="none", edgecolors=color, linewidths=2, zorder=5, label=etq)
ax.set_xlabel("Cobertura (proporción de diagnósticos aceptados)")
ax.set_ylabel("Precisión sobre los aceptados")
ax.set_title("Compromiso entre cobertura y precisión según el umbral de confianza",
             fontweight="bold")
ax.legend(fontsize=8)
save_fig(fig, "fig_cobertura_precision.png"); plt.show()

## 9. Grad-CAM de imágenes mal clasificadas

In [ ]:
pesos = [p for p in glob.glob(os.path.join(CFG.INPUT_ROOT, "**", "best_model.pth"),
                              recursive=True)
         if PRIN in p or PRIN.replace("_", "-") in p]

if not pesos:
    print(f"No se encontraron los pesos de {PRIN}: se omite Grad-CAM")
else:
    import timm
    from torchvision import datasets, transforms
    from pytorch_grad_cam import GradCAM
    from pytorch_grad_cam.utils.image import show_cam_on_image
    from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

    class CBAM(nn.Module):
        def __init__(self, ch, reduction=16):
            super().__init__()
            self.avg_pool = nn.AdaptiveAvgPool2d(1); self.max_pool = nn.AdaptiveMaxPool2d(1)
            self.fc = nn.Sequential(nn.Conv2d(ch, ch // reduction, 1, bias=False), nn.ReLU(),
                                    nn.Conv2d(ch // reduction, ch, 1, bias=False))
            self.sigmoid = nn.Sigmoid()
            self.conv_spa = nn.Conv2d(2, 1, 7, padding=3, bias=False)
        def forward(self, x):
            x = x * self.sigmoid(self.fc(self.avg_pool(x)) + self.fc(self.max_pool(x)))
            a = torch.mean(x, 1, keepdim=True); m, _ = torch.max(x, 1, keepdim=True)
            return x * self.sigmoid(self.conv_spa(torch.cat([a, m], 1)))

    class Clasificador(nn.Module):
        def __init__(self, timm_name, n_cls, usa_cbam, dropout=0.3):
            super().__init__()
            self.usa_cbam = usa_cbam
            self.backbone = timm.create_model(
                timm_name, pretrained=False, num_classes=0,
                **({"global_pool": ""} if usa_cbam else {}))
            with torch.no_grad():
                s = self.backbone(torch.zeros(1, 3, 224, 224))
            self.num_features = s.shape[1]
            if usa_cbam:
                self.cbam = CBAM(self.num_features); self.pool = nn.AdaptiveAvgPool2d(1)
            self.classifier = nn.Sequential(nn.LayerNorm(self.num_features),
                                            nn.Dropout(dropout),
                                            nn.Linear(self.num_features, n_cls))
        def forward(self, x):
            x = self.backbone(x)
            if self.usa_cbam:
                x = self.pool(self.cbam(x)).flatten(1)
            return self.classifier(x)

    cfg_p = R[PRIN]["config"]
    modelo = Clasificador(cfg_p["backbone_timm"], num_classes, cfg_p["usa_cbam"])
    modelo.load_state_dict(torch.load(pesos[0], map_location="cpu"))
    modelo = modelo.to(device).eval()

    val_tf = transforms.Compose([
        transforms.Resize(256), transforms.CenterCrop(224), transforms.ToTensor(),
        transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))])
    test_ds = datasets.ImageFolder(os.path.join(CFG.DATA_DIR, "test"))

    errores = np.where(PRED[PRIN] != y_true)[0]
    errores = errores[np.argsort(-pr_con[errores].max(1))][:CFG.N_GRADCAM]
    print(f"Errores totales: {int((PRED[PRIN] != y_true).sum())} | "
          f"mostrando los {len(errores)} más confiados")

    if cfg_p["usa_cbam"]:
        capas = [modelo.cbam]
    else:
        convs = [m for m in modelo.backbone.modules() if isinstance(m, nn.Conv2d)]
        if not convs:
            raise RuntimeError("Backbone sin convoluciones: elige la capa objetivo a mano")
        capas = [convs[-1]]
    fig, axes = plt.subplots(len(errores), 2,
                             figsize=(7, 3.3 * len(errores)), squeeze=False)
    for fila, i in enumerate(errores):
        ruta, _ = test_ds.samples[i]
        pil = Image.open(ruta).convert("RGB")
        t = val_tf(pil).unsqueeze(0).to(device)
        vista = np.asarray(transforms.CenterCrop(224)(transforms.Resize(256)(pil)),
                           dtype=np.float32) / 255.0
        with GradCAM(model=modelo, target_layers=capas) as cam:
            heat = cam(input_tensor=t,
                       targets=[ClassifierOutputTarget(int(PRED[PRIN][i]))])[0]
        axes[fila, 0].imshow(vista)
        axes[fila, 1].imshow(show_cam_on_image(vista, heat, use_rgb=True))
        axes[fila, 0].set_ylabel(f"real:\n{short_name(class_names[y_true[i]])}",
                                 fontsize=7, rotation=0, ha="right", va="center", labelpad=50)
        axes[fila, 1].set_title(f"predijo {short_name(class_names[PRED[PRIN][i]])} "
                                f"({pr_con[i].max():.0%})", fontsize=8, color="#c75146")
        for a in axes[fila]:
            a.set_xticks([]); a.set_yticks([]); a.grid(False)
    fig.suptitle("Grad-CAM sobre errores de clasificación\n"
                 "¿la atención cae sobre la lesión o sobre el fondo?",
                 fontweight="bold", y=1.0)
    save_fig(fig, "fig_gradcam_errores.png"); plt.show()
    print("\nEn el artículo: clasificar no es localizar. Afirmar que el modelo identifica")
    print("espacialmente las lesiones exigiría regiones anotadas de referencia.")

## 10. Empaquetado

In [ ]:
resumen = {
    "variantes_evaluadas": VARIANTES,
    "n_test": int(len(y_true)),
    "n_test_sin_fuga": int(clean.sum()) if clean is not None else None,
    "ablacion_cbam": ablacion,
    "temperatura": TEMPERATURA,
    "ece_sin_calibrar": float(e_sin) if np.isfinite(e_sin) else None,
    "ece_calibrado": float(e_con) if np.isfinite(e_con) else None,
    "macro_f1_ic95": [float(macro_ic[0]), float(macro_ic[1])],
}
with open(os.path.join(TAB_DIR, "resumen_fase2.json"), "w") as f:
    json.dump(resumen, f, indent=2, ensure_ascii=False, default=float)

lineas = ["# Tablas para el manuscrito", "",
          f"Generadas el {pd.Timestamp.now():%Y-%m-%d %H:%M}. "
          f"{len(y_true)} imágenes de prueba, {num_classes} clases.", ""]

lineas += ["## Tabla A — Comparación de arquitecturas", "",
           "Misma partición congelada, misma receta de entrenamiento, "
           "sin búsqueda de hiperparámetros por arquitectura.", "",
           tabla_arq.round(4).to_markdown(index=False), ""]

if ablacion is not None:
    lineas += ["## Tabla B — Ablación de CBAM", "",
               f"- Accuracy con CBAM: {ablacion['accuracy_cbam']:.4f}",
               f"- Accuracy sin CBAM: {ablacion['accuracy_sin_cbam']:.4f}",
               f"- Diferencia: {ablacion['diferencia_accuracy']:+.4f} "
               f"(IC 95% [{ablacion['ic95'][0]:+.4f}, {ablacion['ic95'][1]:+.4f}])",
               f"- {ablacion['prueba']}, p = {ablacion['p']:.4g}",
               f"- Discordantes: b = {ablacion['b']}, c = {ablacion['c']}"]
    if "macro_f1_dif" in ablacion:
        lineas += [f"- Δ macro-F1 = {ablacion['macro_f1_dif']:+.4f} "
                   f"(IC 95% bootstrap [{ablacion['macro_f1_ic95'][0]:+.4f}, "
                   f"{ablacion['macro_f1_ic95'][1]:+.4f}], p = {ablacion['macro_f1_p']:.4g})"]
    if "sin_fuga" in ablacion:
        sf = ablacion["sin_fuga"]
        lineas += [f"- Sobre el subconjunto sin fuga (n = {sf['n']}): "
                   f"Δ = {sf['dif_accuracy']:+.4f}, p = {sf['p']:.4g}"]
    lineas += [""]

if clean is not None:
    lineas += ["## Tabla C — Resultados con y sin fuga de datos", "",
               tabla_fuga.round(4).to_markdown(index=False), ""]

lineas += ["## Tabla D — Incertidumbre por clase", "",
           "Precision y recall con intervalo de Wilson; F1 con bootstrap. "
           f"Macro-F1 = {macro:.4f} (IC 95% [{macro_ic[0]:.4f}, {macro_ic[1]:.4f}]).", "",
           tabla_clase.drop(columns=["Amplitud IC F1"]).round(4).to_markdown(index=False), ""]

if np.isfinite(e_sin):
    lineas += ["## Calibración", "",
               f"- Temperatura ajustada en validación: {TEMPERATURA:.3f}",
               f"- ECE sin calibrar: {e_sin:.4f}", f"- ECE calibrado: {e_con:.4f}", ""]

with open(os.path.join(TAB_DIR, "tablas_para_el_articulo.md"), "w") as f:
    f.write("\n".join(lineas))
print("tablas_para_el_articulo.md escrito\n")

zip_path = os.path.join(CFG.OUT_DIR, "resultados_fase2.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for carpeta in (FIG_DIR, TAB_DIR):
        for f_ in sorted(os.listdir(carpeta)):
            z.write(os.path.join(carpeta, f_), os.path.join(os.path.basename(carpeta), f_))

print(f"{zip_path}  ({os.path.getsize(zip_path)/1e6:.1f} MB)")
print("\nFiguras:"); [print("  ", f) for f in sorted(os.listdir(FIG_DIR))]
print("Tablas:");    [print("  ", f) for f in sorted(os.listdir(TAB_DIR))]